# Starter 1 — Traditional OCR pipeline, from scratch

> **Status: skeleton.** Structure and interfaces are in place; the model-dependent cells are stubs marked `TODO(kevin)`. Baseline CERs land here before launch.

**Task recap:** given a page image from one of four UW collections (surveyors' field notes, German Kurrent letters, ledger account books, microfilm treaty documents), emit a faithful verbatim transcription. Every model used must be open-weight and under 70B parameters. See the repo README and `docs/transcription_conventions.md`.

This notebook builds the classic three-stage pipeline with no HTR framework, so every stage is visible and swappable:

1. **Layout detection** — find the text regions on the page (and skip sketches/plat maps)
2. **Language routing** — decide per page (or region) whether this is English or German material, and route to the matching recognizer
3. **Recognition** — transcribe each text line, then assemble lines into the page transcription in reading order

Why bother when VLMs exist? Because each stage is independently measurable and independently fixable — when the score is bad you know *which third* of the problem is failing, and a fine-tuned line recognizer can be trained on a laptop.

## 0. Setup

In [ ]:
%pip install -q pillow numpy opencv-python-headless pandas

from pathlib import Path
import numpy as np
from PIL import Image

DATA_DIR = Path("../data/sample")  # or the Kaggle dataset mount, e.g. /kaggle/input/<slug>/test
pages = sorted(DATA_DIR.glob("images/*.jpg"))
print(f"{len(pages)} sample pages found")

## 1. Layout detection

Goal: binarize, find text blocks, and split them into line images. Classic tools go a long way on prose pages; the survey notebooks' drawn tables are where this stage earns its keep (or fails).

Approach sketched here: grayscale → adaptive threshold → horizontal projection profile for line segmentation within detected blocks.

In [ ]:
import cv2

def binarize(img: np.ndarray) -> np.ndarray:
    """Adaptive threshold; archival pages have uneven illumination and fading."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
    return cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY_INV, blockSize=51, C=15)

def segment_lines(binary: np.ndarray, min_height: int = 12) -> list:
    """Horizontal projection profile -> (top, bottom) line bands.
    TODO(kevin): replace with a real layout model for tables/columns; this
    only handles single-column prose."""
    profile = binary.sum(axis=1)
    threshold = profile.max() * 0.05 if profile.max() else 0
    bands, start = [], None
    for y, v in enumerate(profile):
        if v > threshold and start is None:
            start = y
        elif v <= threshold and start is not None:
            if y - start >= min_height:
                bands.append((start, y))
            start = None
    if start is not None:
        bands.append((start, len(profile)))
    return bands

if pages:
    img = np.array(Image.open(pages[0]))
    lines = segment_lines(binarize(img))
    print(f"{pages[0].name}: {len(lines)} candidate lines")

## 2. Language routing

The Kade letters are German (Kurrent script); everything else is English. A cheap router: run a fast recognizer over a few lines and score the output against English/German character n-gram profiles — or at page level, use collection metadata when available. The test set's `metadata.csv` gives you `category`, which determines language exactly — so at inference time routing is trivial; this stage matters if you want a pipeline that generalizes past this challenge.

In [ ]:
def route_language(category: str) -> str:
    """Trivial router for the challenge; TODO(kevin): n-gram router for the general case."""
    return "de" if category == "kade_letters" else "en"

print(route_language("kade_letters"), route_language("survey_notes"))

## 3. Recognition

The from-scratch option: a small CRNN/CTC line recognizer trained on line images + transcripts (IAM for plumbing, Bentham/Escher for period hands — see `RESOURCES.md`). This is the stage that most rewards fine-tuning on self-curated samples.

In [ ]:
def recognize_line(line_img: np.ndarray, lang: str) -> str:
    """TODO(kevin): CRNN/CTC line recognizer (train on IAM to validate the loop,
    then Bentham for English hands / Escher for Kurrent)."""
    raise NotImplementedError

def transcribe_page(page_path: Path, category: str) -> str:
    img = np.array(Image.open(page_path))
    binary = binarize(img)
    lang = route_language(category)
    out = []
    for top, bottom in segment_lines(binary):
        out.append(recognize_line(img[top:bottom], lang))
    return "\n".join(out)

## Scoring your output

Whatever pipeline you build, score it the way the leaderboard will — verbatim CER with whitespace collapsed, macro-averaged by category. `evaluation/metric.py` in the challenge repo is the single source of truth.

In [ ]:
import sys
sys.path.insert(0, "../evaluation")  # adjust if running on Kaggle: attach the challenge repo as a dataset
from metric import page_cer, page_wer

prediction = "North between Sections 13 & 14"
reference  = "North between Sections 13 & 14"
print("CER:", page_cer(prediction, reference), " WER:", page_wer(prediction, reference))

## 4. Build a submission

Run `transcribe_page` over every row of the test `metadata.csv` and write `page_id,text` — then check it with `evaluation/score_local.py` against your own calibration set before spending a daily submission.

In [ ]:
import pandas as pd

# meta = pd.read_csv(DATA_DIR / "metadata.csv")
# sub = pd.DataFrame({
#     "page_id": meta["page_id"],
#     "text": [transcribe_page(DATA_DIR / "images" / f"{pid}.jpg", cat)
#              for pid, cat in zip(meta["page_id"], meta["category"])],
# })
# sub.to_csv("submission.csv", index=False)

## Ideas to beat this baseline

- Replace projection-profile segmentation with a learned layout model — the drawn tables in the survey notebooks will break the naive version
- Fine-tune the recognizer per category (~100 curated lines can move a category a lot)
- A language-model rescoring pass over recognizer lattices — but remember: *verbatim* scoring means an LM that modernizes spelling makes things worse
- Table-aware reading order for the Dominy ledgers